In [4]:
# ============================================================
# CELL 0: TẠO DATASET RIÊNG CHO STANDARD (ham/scam only)
# Không ghi đè file gốc — tạo file mới cleaned_dataset_standard.csv
# ============================================================

import os, re, glob
import pandas as pd
from kaggle_secrets import UserSecretsClient

# ============================================================
# 1. LOAD FILE GỐC
# ============================================================
csv_paths = glob.glob('/kaggle/input/**/cleaned_dataset.csv', recursive=True)
if not csv_paths:
    raise FileNotFoundError("Không tìm thấy cleaned_dataset.csv")

df_raw = pd.read_csv(csv_paths[0]).dropna(subset=['content', 'label'])
print(f"✅ Đọc file gốc: {csv_paths[0]}")
print(f"   Tổng rows: {len(df_raw)}")
print(f"   Label gốc:\n{df_raw['label'].value_counts()}\n")

# ============================================================
# 2. LỌC CHỈ GIỮ ham + scam
# other → ham, bỏ spam hoàn toàn
# ============================================================
df = df_raw.copy()
df['label'] = df['label'].astype(str).str.lower().str.strip()
df['label'] = df['label'].replace('other', 'ham')
df = df[df['label'].isin(['ham', 'scam'])].copy()
df = df.reset_index(drop=True)

print(f"   Sau lọc (chỉ ham/scam): {len(df)} rows")
print(f"   Label mới:\n{df['label'].value_counts()}\n")

# ============================================================
# 3. LƯU FILE MỚI — không đụng file gốc
# ============================================================
OUTPUT_PATH = '/kaggle/working/cleaned_dataset_standard.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Đã lưu dataset Standard: {OUTPUT_PATH}")
print(f"   Columns: {list(df.columns)}")
print(f"\nPreview:")
print(df.head(3).to_string())

✅ Đọc file gốc: /kaggle/input/datasets/khoiho1234/cleaned-data-csv/cleaned_dataset.csv
   Tổng rows: 5366
   Label gốc:
label
scam     1816
spam     1784
ham      1765
label       1
Name: count, dtype: int64

   Sau lọc (chỉ ham/scam): 3581 rows
   Label mới:
label
scam    1816
ham     1765
Name: count, dtype: int64

✅ Đã lưu dataset Standard: /kaggle/working/cleaned_dataset_standard.csv
   Columns: ['label', 'content']

Preview:
  label                                                                             content
0  scam         Để có cơ hội để giành được £ 250 wkly mua sắm spree txt: mua sắm lên 80878.
1  scam  Shopee Pay: Số dư ví 47 triệu đang chờ rút. Click xác thực khuôn mặt để nhận tiền.
2   ham                                Aight Tôi có nên lên kế hoạch đến sau tối nay không?


In [5]:
# ============================================================
# CELL 1: TRAIN FASTTEXT TRÊN KAGGLE (thay thế BlazingText)
# ============================================================

# ── PATCH numpy trong fasttext module (numpy 2.x compat) ──
import fasttext.FastText as _ft_module, numpy as _np
_orig_np_array = _np.array
def _ft_safe_array(obj, *args, copy=None, **kwargs):
    if copy is False:
        return _np.asarray(obj, *args, **kwargs)
    return _orig_np_array(obj, *args, **kwargs)
_ft_module.np.array = _ft_safe_array
# ──────────────────────────────────────────────────────────

import os, re, glob, json
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from collections import Counter

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_INPUT = os.path.join(OUTPUT_DIR, 'cleaned_dataset_standard.csv')
if not os.path.exists(DATA_INPUT):
    csv_paths = glob.glob('/kaggle/input/**/cleaned_dataset.csv', recursive=True)
    if not csv_paths:
        raise FileNotFoundError("Không tìm thấy cleaned_dataset.csv")
    df = pd.read_csv(csv_paths[0]).dropna(subset=['content', 'label'])
    df['label'] = df['label'].astype(str).str.lower().str.strip()
    df['label'] = df['label'].replace('other', 'ham')
    df = df[df['label'].isin(['ham', 'scam'])].copy().reset_index(drop=True)
else:
    df = pd.read_csv(DATA_INPUT).dropna(subset=['content', 'label'])

print(f"Data: {len(df)} rows")
print(df['label'].value_counts())

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+', ' url ', text)
    text = re.sub(r'(0\d{9,10})', ' phone ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['content'].apply(clean_text)
df['bt_label']   = '__label__' + df['label']

dist         = Counter(df['label'])
majority_lbl = max(dist, key=dist.get)
minority_lbl = min(dist, key=dist.get)
ratio        = dist[majority_lbl] / dist[minority_lbl]

if ratio > 2.0:
    target_count = int(dist[majority_lbl] * 0.85)
    minority_df  = df[df['label'] == minority_lbl]
    oversampled  = minority_df.sample(target_count, replace=True, random_state=42)
    df = pd.concat([df, oversampled]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Sau oversample: {Counter(df['label'])}")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_path = os.path.join(OUTPUT_DIR, 'train_standard.txt')
test_path  = os.path.join(OUTPUT_DIR, 'test_standard.txt')

train_df[['bt_label', 'clean_text']].to_csv(train_path, index=False, header=False, sep=' ')
test_df[['bt_label',  'clean_text']].to_csv(test_path,  index=False, header=False, sep=' ')
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

print("\n🚀 Bắt đầu train fasttext...")
model = fasttext.train_supervised(
    input      = train_path,
    epoch      = 30,
    lr         = 0.05,
    wordNgrams = 3,
    dim        = 100,
    minCount   = 1,
    bucket     = 2000000,
    loss       = 'softmax',
)

print("\n📊 Đánh giá trên test set:")
result = model.test(test_path)
print(f"  Samples: {result[0]} | Precision: {result[1]:.4f} | Recall: {result[2]:.4f}")

preds, truths = [], []
with open(test_path) as f:
    for line in f:
        parts = line.strip().split(' ', 1)
        if len(parts) < 2:
            continue
        truth = parts[0].replace('__label__', '')
        labels, _ = model.predict(parts[1])
        pred  = labels[0].replace('__label__', '')
        truths.append(truth)
        preds.append(pred)

print(classification_report(truths, preds, target_names=['ham', 'scam'], digits=4))

model_path = os.path.join(OUTPUT_DIR, 'model_standard.bin')
model.save_model(model_path)
print(f"✅ Model lưu tại: {model_path}")

Data: 3581 rows
label
scam    1816
ham     1765
Name: count, dtype: int64
Train: 2864 | Test: 717

🚀 Bắt đầu train fasttext...


Read 0M words
Number of words:  5416
Number of labels: 2
Progress: 100.0% words/sec/thread:  638019 lr:  0.000000 avg.loss:  0.206380 ETA:   0h 0m 0s



📊 Đánh giá trên test set:
  Samples: 717 | Precision: 0.9191 | Recall: 0.9191
              precision    recall  f1-score   support

         ham     0.9132    0.9235    0.9183       353
        scam     0.9250    0.9148    0.9199       364

    accuracy                         0.9191       717
   macro avg     0.9191    0.9192    0.9191       717
weighted avg     0.9192    0.9191    0.9191       717

✅ Model lưu tại: /kaggle/working/model_standard.bin


In [6]:
# ============================================================
# CELL 2: ĐÓNG GÓI MODEL + UPLOAD S3
# ============================================================

import os, tarfile, boto3
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID']     = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ['AWS_SECRET_ACCESS_KEY'] = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ['AWS_DEFAULT_REGION']    = user_secrets.get_secret("AWS_DEFAULT_REGION")

BUCKET     = user_secrets.get_secret("S3_BUCKET_NAME")
REGION     = os.environ['AWS_DEFAULT_REGION']
OUTPUT_DIR = '/kaggle/working'

model_bin  = os.path.join(OUTPUT_DIR, 'model_standard.bin')
tar_path   = os.path.join(OUTPUT_DIR, 'model_standard.tar.gz')

if not os.path.exists(model_bin):
    raise FileNotFoundError("Chạy Cell 1 trước!")

# Đóng gói — SageMaker BlazingText cần file tên "model" bên trong tar
print("📦 Đóng gói model.tar.gz...")
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(model_bin, arcname='model.bin')

size_mb = os.path.getsize(tar_path) / 1024 / 1024
print(f"✅ Đóng gói xong: {tar_path} ({size_mb:.1f} MB)")

# Upload S3
s3_key = 'standard/output/fasttext/model_standard.tar.gz'
print(f"⏳ Uploading lên s3://{BUCKET}/{s3_key}...")
s3 = boto3.client('s3', region_name=REGION)
s3.upload_file(tar_path, BUCKET, s3_key)
print(f"✅ Upload xong!")

# Lưu path để Cell 3 dùng
import json
config = {'model_s3_path': f's3://{BUCKET}/{s3_key}'}
with open(os.path.join(OUTPUT_DIR, 'standard_model_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print(f"Model S3 path: {config['model_s3_path']}")

📦 Đóng gói model.tar.gz...
✅ Đóng gói xong: /kaggle/working/model_standard.tar.gz (231.3 MB)
⏳ Uploading lên s3://spam-detection-doannhom/standard/output/fasttext/model_standard.tar.gz...
✅ Upload xong!
Model S3 path: s3://spam-detection-doannhom/standard/output/fasttext/model_standard.tar.gz


In [7]:
# ============================================================
# CELL 3: DEPLOY ENDPOINT TỪ FASTTEXT MODEL
# ============================================================

import os, json, time, boto3
from botocore.config import Config
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID']     = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ['AWS_SECRET_ACCESS_KEY'] = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ['AWS_DEFAULT_REGION']    = user_secrets.get_secret("AWS_DEFAULT_REGION")

REGION        = os.environ['AWS_DEFAULT_REGION']
ROLE          = 'arn:aws:iam::992409270804:role/SageMakerExecutionRole'
CONTAINER     = '475088953585.dkr.ecr.ap-southeast-1.amazonaws.com/blazingtext:1'
ENDPOINT_NAME = 'spam-detection-standard'
BOTO_CONFIG   = Config(connect_timeout=15, read_timeout=120, retries={'max_attempts': 3, 'mode': 'adaptive'})

config_path = '/kaggle/working/standard_model_config.json'
if not os.path.exists(config_path):
    raise FileNotFoundError("Chạy Cell 2 trước!")

with open(config_path) as f:
    model_s3_path = json.load(f)['model_s3_path']
print(f"Model S3: {model_s3_path}")

sm = boto3.client('sagemaker', region_name=REGION, config=BOTO_CONFIG)

# Xóa tài nguyên cũ nếu có
for label, fn, kwargs in [
    ('Endpoint',       sm.delete_endpoint,        {'EndpointName':       ENDPOINT_NAME}),
    ('EndpointConfig', sm.delete_endpoint_config, {'EndpointConfigName': ENDPOINT_NAME}),
    ('Model',          sm.delete_model,           {'ModelName':          ENDPOINT_NAME}),
]:
    try:
        fn(**kwargs)
        print(f"Đã xóa {label} cũ")
        time.sleep(3)
    except Exception:
        pass

# Tạo Model
sm.create_model(
    ModelName        = ENDPOINT_NAME,
    PrimaryContainer = {'Image': CONTAINER, 'ModelDataUrl': model_s3_path},
    ExecutionRoleArn = ROLE
)
print(f"✅ Model created")

# Tạo EndpointConfig
sm.create_endpoint_config(
    EndpointConfigName = ENDPOINT_NAME,
    ProductionVariants = [{
        'VariantName':          'AllTraffic',
        'ModelName':            ENDPOINT_NAME,
        'InitialInstanceCount': 1,
        'InstanceType':         'ml.t2.medium'
    }]
)
print(f"✅ EndpointConfig created")

# Deploy
sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_NAME)
print(f"⏳ Đang deploy {ENDPOINT_NAME}...")

while True:
    status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)['EndpointStatus']
    print(f"Status: {status}")
    if status == 'InService':
        print(f"\n✅ Endpoint [{ENDPOINT_NAME}] sẵn sàng!")
        break
    if status in ('Failed', 'RollingBack'):
        raise RuntimeError(f"Deploy thất bại: {status}")
    time.sleep(20)

Model S3: s3://spam-detection-doannhom/standard/output/fasttext/model_standard.tar.gz
✅ Model created
✅ EndpointConfig created
⏳ Đang deploy spam-detection-standard...
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService

✅ Endpoint [spam-detection-standard] sẵn sàng!


In [9]:
# ============================================================
# CELL 4: XÓA TẤT CẢ ENDPOINT ĐANG INSERVICE
# Chỉ xóa những thứ đang chạy — không xóa S3, model artifacts
# ============================================================

import os, boto3, time
from botocore.config import Config
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID']     = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ['AWS_SECRET_ACCESS_KEY'] = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ['AWS_DEFAULT_REGION']    = user_secrets.get_secret("AWS_DEFAULT_REGION")

REGION      = os.environ['AWS_DEFAULT_REGION']
BOTO_CONFIG = Config(connect_timeout=15, read_timeout=120, retries={'max_attempts': 3, 'mode': 'adaptive'})
sm          = boto3.client('sagemaker', region_name=REGION, config=BOTO_CONFIG)

# ============================================================
# 1. LẤY DANH SÁCH TẤT CẢ ENDPOINT ĐANG CHẠY
# ============================================================
all_endpoints = []
paginator = sm.get_paginator('list_endpoints')
for page in paginator.paginate():
    all_endpoints.extend(page['Endpoints'])

active = [e for e in all_endpoints if e['EndpointStatus'] in ('InService', 'Creating', 'Updating')]
print(f"Tìm thấy {len(active)} endpoint đang chạy:")
for e in active:
    print(f"  🔴 {e['EndpointName']} — {e['EndpointStatus']}")

if not active:
    print("✅ Không có endpoint nào đang chạy.")
else:
    print()
    for e in active:
        name = e['EndpointName']
        print(f"{'='*50}")
        print(f"🗑️  Xóa: {name}")

        # Xóa Endpoint — chờ xong mới xóa config/model
        try:
            sm.delete_endpoint(EndpointName=name)
            print(f"  ⏳ Đang xóa Endpoint...")
            waiter = sm.get_waiter('endpoint_deleted')
            waiter.wait(EndpointName=name, WaiterConfig={'Delay': 10, 'MaxAttempts': 30})
            print(f"  ✅ Endpoint đã xóa")
        except Exception as e:
            print(f"  ❌ Endpoint: {e}")

        # Xóa EndpointConfig
        try:
            sm.delete_endpoint_config(EndpointConfigName=name)
            print(f"  ✅ EndpointConfig đã xóa")
        except Exception as e:
            print(f"  ⚠️  EndpointConfig: {e}")

        # Xóa Model
        try:
            sm.delete_model(ModelName=name)
            print(f"  ✅ Model đã xóa")
        except Exception as e:
            print(f"  ⚠️  Model: {e}")

# ============================================================
# 2. KIỂM TRA TRAINING JOB ĐANG CHẠY
# ============================================================
print(f"\n{'='*50}")
jobs = sm.list_training_jobs(StatusEquals='InProgress')['TrainingJobSummaries']
if jobs:
    print(f"⚠️  Có {len(jobs)} training job đang chạy:")
    for j in jobs:
        print(f"  🔴 {j['TrainingJobName']} — dùng lệnh stop nếu muốn dừng")
else:
    print("✅ Không có training job nào đang chạy.")

print(f"\n✅ Hoàn tất. SageMaker không còn tính phí runtime.")
print(f"   S3 và model artifacts vẫn được giữ nguyên.")

Tìm thấy 0 endpoint đang chạy:
✅ Không có endpoint nào đang chạy.

✅ Không có training job nào đang chạy.

✅ Hoàn tất. SageMaker không còn tính phí runtime.
   S3 và model artifacts vẫn được giữ nguyên.


In [2]:
import subprocess, os

subprocess.run([
    "pip", "install", 
    "fasttext-wheel", 
    "--no-cache-dir", 
    "-t", "/tmp/python311",
    "--python-version", "3.11",
    "--only-binary=:all:",
    "--platform", "manylinux2014_x86_64"
], check=True)

import glob
so_files = glob.glob('/tmp/python311/**/*.so', recursive=True)
print('\n'.join(so_files))
# Cài đúng môi trường
subprocess.run(["pip", "install", "fasttext-wheel", "--no-cache-dir", 
                "-t", "/tmp/python"], check=True)

# Kiểm tra có .so chưa
result = subprocess.run(["find", "/tmp/python", "-name", "*.so"], 
                        capture_output=True, text=True)
print(result.stdout)

# Zip lại
subprocess.run(["zip", "-r9", "/tmp/fasttext-layer-fixed.zip", "python/"], 
               cwd="/tmp", check=True)

print("Size:", os.path.getsize("/tmp/fasttext-layer-fixed.zip") / 1024 / 1024, "MB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 374.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 277.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
moviepy 1.

/tmp/python311/fasttext_pybind.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy.libs/libscipy_openblas64_-56d6093b.so
/tmp/python311/numpy/linalg/_umath_linalg.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/linalg/lapack_lite.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_bounded_integers.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_pcg64.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_sfc64.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_common.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_philox.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/bit_generator.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/mtrand.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_generator.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/random/_mt19937.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/fft/_pocketfft_umath.cpython-311-x86_64-linux-gnu.so
/tmp/python311/numpy/_core/_r

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires 

/tmp/python/fasttext_pybind.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy.libs/libscipy_openblas64_-32a4b2a6.so
/tmp/python/numpy/linalg/_umath_linalg.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/linalg/lapack_lite.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_common.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/bit_generator.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_generator.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_mt19937.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_bounded_integers.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_sfc64.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_philox.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/_pcg64.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/random/mtrand.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/fft/_pocketfft_umath.cpython-312-x86_64-linux-gnu.so
/tmp/python/numpy/_core/_operand_flag_tests.cpython-312-x86_64-linux-gn